In [ ]:
import json
import re
import numpy as np
import pandas as pd

# ----------------------------------------------------
# 1. データの読み込みと辞書化
# ----------------------------------------------------
csv_file_path = 'data/EN_Card_Data.csv'
df_cards = pd.read_csv(csv_file_path)

df_cards['HP'] = df_cards['HP'].fillna(0).astype(float)
df_cards['Retreat'] = df_cards['Retreat'].fillna(0).astype(float)
df_cards['Type'] = df_cards['Type'].fillna("None")
df_cards['Weakness'] = df_cards['Weakness'].fillna("None")
df_cards['Damage'] = df_cards['Damage'].fillna("0")
df_cards['Cost'] = df_cards['Cost'].fillna("")
df_cards['Rule'] = df_cards['Rule'].fillna("")

df_cards_unique = df_cards.drop_duplicates(subset=['Card ID'], keep='first')
card_dict = df_cards_unique.set_index('Card ID').to_dict(orient='index')
MAX_CARD_ID = int(df_cards['Card ID'].max())

TYPE_VOCAB = ['{G}', '{R}', '{W}', '{L}', '{P}', '{F}', '{D}', '{M}', '{C}', '竜', '{A}', '{A}{A}', '{Team Rocket}{Team Rocket}', '{C}{C}{C}']
WEAKNESS_VOCAB = ['{G}', '{R}', '{W}', '{L}', '{P}', '{F}', '{D}', '{M}', '{C}', '竜']

def one_hot_encode(val, vocab):
    vector = [0] * len(vocab)
    if val in vocab:
        vector[vocab.index(val)] = 1
    return vector

def parse_damage(damage_str):
    nums = re.findall(r'\d+', str(damage_str))
    return int(nums[0]) if nums else 0

def parse_cost(cost_str):
    cost_str = str(cost_str)
    return cost_str.count('{') + cost_str.count('●')

def encode_card_list(card_list, max_id):
    vector = [0] * (max_id + 1)
    if not card_list:
        return vector
    for card in card_list:
        card_id = card.get('id', 0)
        if 0 <= card_id <= max_id:
            vector[card_id] += 1
    return vector

def count_attached_energy_types(energy_cards, card_dict):
    counts = [0] * len(TYPE_VOCAB)
    for ec in energy_cards:
        eid = ec.get('id', 0)
        etype = card_dict.get(eid, {}).get('Type', 'None')
        if etype in TYPE_VOCAB:
            counts[TYPE_VOCAB.index(etype)] += 1
    return counts

# ----------------------------------------------------
# 2. 特徴量抽出関数
# ----------------------------------------------------
def extract_detailed_pokemon_features(poke_data, card_dict):
    empty_features = [0] * (7 + 2 + len(TYPE_VOCAB) + len(WEAKNESS_VOCAB) + len(TYPE_VOCAB))
    if not poke_data:
        return empty_features

    card_id = poke_data.get('id', 0)
    current_hp = poke_data.get('hp', 0)
    energy_count = len(poke_data.get('energies', []))

    tools = poke_data.get('tools', [])
    tool_id = tools[0].get('id', 0) if tools else 0

    card_info = card_dict.get(card_id, {})
    max_hp = card_info.get('HP', 0)
    retreat_cost = card_info.get('Retreat', 0)
    attack_damage = parse_damage(card_info.get('Damage', "0"))
    attack_cost = parse_cost(card_info.get('Cost', ""))
    is_ex = 1 if "ex" in str(card_info.get('Rule', "")).lower() else 0

    type_vec = one_hot_encode(card_info.get('Type', 'None'), TYPE_VOCAB)
    weakness_vec = one_hot_encode(card_info.get('Weakness', 'None'), WEAKNESS_VOCAB)

    energy_cards = poke_data.get('energyCards', [])
    attached_energy_types = count_attached_energy_types(energy_cards, card_dict)

    basic_stats = [card_id, current_hp, max_hp, energy_count, retreat_cost, is_ex, tool_id]
    attack_stats = [attack_damage, attack_cost]

    return basic_stats + attack_stats + type_vec + weakness_vec + attached_energy_types

def extract_detailed_player_features(player_data, card_dict, max_card_id):
    features = []

    active_poke = player_data['active'][0] if player_data.get('active') else None
    features.extend(extract_detailed_pokemon_features(active_poke, card_dict))

    bench_pokes = player_data.get('bench', [])
    for i in range(5):
        if i < len(bench_pokes):
            features.extend(extract_detailed_pokemon_features(bench_pokes[i], card_dict))
        else:
            features.extend(extract_detailed_pokemon_features(None, card_dict))

    status_vec = [
        int(player_data.get('poisoned', False)),
        int(player_data.get('burned', False)),
        int(player_data.get('asleep', False)),
        int(player_data.get('paralyzed', False)),
        int(player_data.get('confused', False))
    ]
    features.extend(status_vec)

    hand_count = player_data.get('handCount', 0)
    deck_count = player_data.get('deckCount', 0)
    prize_count = sum(1 for p in player_data.get('prize', []) if p is not None)
    features.extend([hand_count, deck_count, prize_count])

    hand_features = encode_card_list(player_data.get('hand', []), max_card_id)
    discard_features = encode_card_list(player_data.get('discard', []), max_card_id)

    features.extend(hand_features)
    features.extend(discard_features)

    return features

def extract_detailed_state_vector(step_data, card_dict, max_card_id):
    obs = step_data.get('observation', {})
    current_state = obs.get('current')
    if not current_state or len(current_state.get('players', [])) < 2:
        return None

    stadium_list = current_state.get('stadium', [])
    stadium_id = stadium_list[0].get('id', 0) if stadium_list else 0
    supporter_played = int(current_state.get('supporterPlayed', False))
    energy_attached = int(current_state.get('energyAttached', False))
    retreated = int(current_state.get('retreated', False))
    first_player = int(current_state.get('firstPlayer', -1))
    turn_num = int(current_state.get('turn', 0))

    global_features = [stadium_id, supporter_played, energy_attached, retreated, first_player, turn_num]

    my_index = current_state.get('yourIndex', 0)
    op_index = 1 - my_index

    my_features = extract_detailed_player_features(current_state['players'][my_index], card_dict, max_card_id)
    op_features = extract_detailed_player_features(current_state['players'][op_index], card_dict, max_card_id)

    return np.array(global_features + my_features + op_features, dtype=np.float32)

# ----------------------------------------------------
# 3. 状態ベクトルの次元数を確認
# ----------------------------------------------------
import os
archive_dir = 'archive'
sample_file = os.path.join(archive_dir, os.listdir(archive_dir)[0])

with open(sample_file, 'r') as f:
    battle_log = json.load(f)

expected_player_features_len = 47 + (5 * 47) + 5 + 3 + (MAX_CARD_ID + 1) + (MAX_CARD_ID + 1)
expected_state_vector_len = 6 + (2 * expected_player_features_len)

print(f"MAX_CARD_ID: {MAX_CARD_ID}")
print(f"プレイヤー特徴量: {expected_player_features_len} 次元")
print(f"状態ベクトル合計: {expected_state_vector_len} 次元")

for step_idx, step_data_list in enumerate(battle_log['steps']):
    for player_idx, player_step_data in enumerate(step_data_list):
        state_vector = extract_detailed_state_vector(player_step_data, card_dict, MAX_CARD_ID)
        if state_vector is not None:
            if len(state_vector) != expected_state_vector_len:
                print(f"Warning: step {step_idx}, player {player_idx}: {len(state_vector)} (expected {expected_state_vector_len})")

print(f"確認完了 (サンプルファイル: {os.path.basename(sample_file)})")


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PTCGBaselineNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=1024, max_actions=256):
        """
        引数:
            input_dim: 入力特徴量ベクトルの次元数 (約4400)
            hidden_dim: 隠れ層のノード数
            max_actions: 出力する最大行動候補数
        """
        super(PTCGBaselineNet, self).__init__()

        # 第1層 (入力層 -> 隠れ層1)
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.ln1 = nn.LayerNorm(hidden_dim) # 学習を安定させる正規化層

        # 第2層 (隠れ層1 -> 隠れ層2)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.ln2 = nn.LayerNorm(hidden_dim // 2)

        # 出力層 (隠れ層2 -> 各行動のスコア)
        # ※ポケカではそのターンに選べる選択肢が変動するため、
        # 十分に大きい固定長(例:256)を出力し、後でマスク処理を行います。
        self.fc3 = nn.Linear(hidden_dim // 2, max_actions)

    def forward(self, x):
        """
        順伝播処理 (データを流し込む処理)
        x の形状: (バッチサイズ, input_dim)
        """
        # ReLU活性化関数で非線形性を加える
        x = F.relu(self.ln1(self.fc1(x)))
        x = F.relu(self.ln2(self.fc2(x)))

        # 最終出力 (Logits: 確率に変換される前の生スコア)
        logits = self.fc3(x)

        return logits

# ----------------------------------------------------
# 動作確認 (テスト実行)
# ----------------------------------------------------
# 仮の特徴量次元数 (前回のコードで出力された次元数を指定してください)
INPUT_DIM = 5658 # 修正: 状態ベクトルの次元数を5658に更新
MAX_ACTIONS = 256 # ポケカの1回のプロンプトで提示される選択肢の最大値を想定

# モデルのインスタンス化
model = PTCGBaselineNet(input_dim=INPUT_DIM, max_actions=MAX_ACTIONS)

# ダミーの入力データ (バッチサイズ1, 特徴量次元数INPUT_DIM) を作成
dummy_state_vector = torch.randn(1, INPUT_DIM)

# モデルに推論させる
output_logits = model(dummy_state_vector)

print("==== ネットワーク構造 ====")
print(model)
print("\n==== 出力の形状 ====")
print(f"入力形状: {dummy_state_vector.shape}")
print(f"出力形状: {output_logits.shape} -> (バッチサイズ, 最大行動数)")

==== ネットワーク構造 ====
PTCGBaselineNet(
  (fc1): Linear(in_features=5658, out_features=1024, bias=True)
  (ln1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  (fc2): Linear(in_features=1024, out_features=512, bias=True)
  (ln2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (fc3): Linear(in_features=512, out_features=256, bias=True)
)

==== 出力の形状 ====
入力形状: torch.Size([1, 5658])
出力形状: torch.Size([1, 256]) -> (バッチサイズ, 最大行動数)


In [10]:
import torch
import torch.nn.functional as F

# --- (前回のモデル定義・ダミー出力がすでにある前提です) ---

# 1. 現在のステップで「選べる行動の数」を取得する
# 実際のゲーム中では、obs.select.option の長さになります
# ここではテストとして、仮に「今は7個の行動しか選べない」状況だとします
valid_action_count = 7

# 2. マスクの作成
# 0から255までのインデックスを作り、有効な行動数以上のインデックスをTrue(無効)にする
# 例: [False, False, ..., False, True, True, ...] (最初の7個がFalse、残りがTrue)
mask = torch.arange(MAX_ACTIONS) >= valid_action_count

# バッチサイズ(1)に合わせて次元を追加: shape -> (1, 256)
mask = mask.unsqueeze(0)

print(f"有効な行動数: {valid_action_count} 個")
print(f"適用前の先頭10個のスコア:\n{output_logits[0, :10].detach().numpy()}")

# 3. マスクの適用 (Masked Fill)
# maskがTrueの場所（無効な行動）のスコアを -1e9 (非常に小さな値) に書き換える
masked_logits = output_logits.masked_fill(mask, -1e9)

print(f"\n適用後の先頭10個のスコア (8個目以降が -1e9 になります):\n{masked_logits[0, :10].detach().numpy()}")

# 4. 確率への変換と行動の決定
# ソフトマックス関数を通すことで、-1e9のスコアは確率「0.0」になります
action_probs = F.softmax(masked_logits, dim=-1)

# 最も確率(スコア)が高い行動のインデックスを取得
best_action_index = torch.argmax(masked_logits, dim=-1).item()

print(f"\n各行動の選択確率 (先頭10個):\n{action_probs[0, :10].detach().numpy()}")
print(f"👉 AIが最終的に選択した行動インデックス: {best_action_index}")

有効な行動数: 7 個
適用前の先頭10個のスコア:
[-0.28114617  0.21537194  0.07722566  0.4399745   0.35139224 -0.10659562
  0.29054153  0.38120624 -0.0363941  -0.74813557]

適用後の先頭10個のスコア (8個目以降が -1e9 になります):
[-2.8114617e-01  2.1537194e-01  7.7225655e-02  4.3997449e-01
  3.5139224e-01 -1.0659562e-01  2.9054153e-01 -1.0000000e+09
 -1.0000000e+09 -1.0000000e+09]

各行動の選択確率 (先頭10個):
[0.09111557 0.14970204 0.13038616 0.18740076 0.17151439 0.10849231
 0.16138881 0.         0.         0.        ]
👉 AIが最終的に選択した行動インデックス: 3


In [ ]:
import os
import glob
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

try:
    from tqdm.auto import tqdm
except ImportError:
    # tqdmがない環境向けのフォールバック
    def tqdm(iterable, **kwargs):
        desc = kwargs.get('desc', '')
        total = kwargs.get('total', None)
        print(f"{desc} (tqdm未インストール、インストール推奨: pip install tqdm)")
        return iterable

# ============================================================
# 設定パラメータ（ここを変更して実験）
# ============================================================
JSON_DIRECTORY = 'archive'       # アーカイブディレクトリ
MAX_ACTIONS    = 256             # 1ターンの最大選択肢数
MAX_FILES      = None            # None=全ファイル / 整数=テスト用に制限 (例: 200)
BATCH_SIZE     = 256
EPOCHS         = 10
LEARNING_RATE  = 1e-3
MODEL_PATH     = 'ptcg_baseline_model.pth'
NORM_PATH      = 'ptcg_normalization.npz'  # 正規化パラメータを保存
# ============================================================

INPUT_DIM = expected_state_vector_len  # Cell 0 で計算済み

# ----------------------------------------------------
# Dataset
# ----------------------------------------------------
class PTCGLogDataset(Dataset):
    def __init__(self, json_dir, card_dict, max_card_id, max_actions=256, max_files=None):
        raw_states  = []
        raw_masks   = []
        raw_targets = []

        json_files = sorted(glob.glob(os.path.join(json_dir, "*.json")))
        if max_files is not None:
            json_files = json_files[:max_files]

        print(f"解析対象: {len(json_files)} ファイル")

        skipped_invalid = 0
        skipped_no_state = 0

        for file_path in tqdm(json_files, desc="JSONパース中"):
            try:
                with open(file_path, 'r') as f:
                    battle_log = json.load(f)
            except Exception as e:
                continue

            rewards = battle_log.get('rewards', [0, 0])
            if rewards[0] == 1:
                winner_index = 0
            elif rewards[1] == 1:
                winner_index = 1
            else:
                continue  # 引き分けはスキップ

            for step in battle_log.get('steps', []):
                if len(step) <= winner_index:
                    continue

                player_step_data = step[winner_index]

                state_vec = extract_detailed_state_vector(player_step_data, card_dict, max_card_id)
                if state_vec is None:
                    skipped_no_state += 1
                    continue

                selected = player_step_data.get('action')
                if not selected:
                    continue

                obs = player_step_data.get('observation', {})
                select_data = obs.get('select')
                if not select_data or 'option' not in select_data:
                    continue

                valid_count = len(select_data['option'])
                target_action = selected[0]

                if target_action >= max_actions or target_action >= valid_count:
                    skipped_invalid += 1
                    continue

                mask_vec = np.arange(max_actions) >= valid_count

                raw_states.append(state_vec)
                raw_masks.append(mask_vec)
                raw_targets.append(target_action)

        print(f"スキップ (state=None): {skipped_no_state}, スキップ (不正action): {skipped_invalid}")
        print(f"有効サンプル数 (正規化前): {len(raw_states)}")

        if not raw_states:
            print("ERROR: 有効サンプルが0件です。パスを確認してください。")
            self.samples = []
            return

        # --- 正規化 ---
        all_states = np.stack(raw_states, axis=0)  # (N, D)
        self.mean = all_states.mean(axis=0)
        self.std  = all_states.std(axis=0)
        self.std[self.std == 0] = 1e-8

        np.savez(NORM_PATH, mean=self.mean, std=self.std)
        print(f"正規化パラメータを保存: {NORM_PATH}")

        norm_states = (all_states - self.mean) / self.std

        self.states  = torch.tensor(norm_states,         dtype=torch.float32)
        self.masks   = torch.tensor(np.stack(raw_masks), dtype=torch.bool)
        self.targets = torch.tensor(raw_targets,         dtype=torch.long)

        print(f"有効サンプル数 (正規化後): {len(self.states)}")

    def __len__(self):
        return len(self.states)

    def __getitem__(self, idx):
        return self.states[idx], self.masks[idx], self.targets[idx]


# ----------------------------------------------------
# データセット作成
# ----------------------------------------------------
print("== データ読み込み開始 ==")
dataset = PTCGLogDataset(JSON_DIRECTORY, card_dict, MAX_CARD_ID,
                         max_actions=MAX_ACTIONS, max_files=MAX_FILES)

if len(dataset) == 0:
    raise RuntimeError("有効サンプルなし。JSON_DIRECTORY のパスと archive/ の内容を確認してください。")

# 学習/検証 8:2 分割
val_size   = int(len(dataset) * 0.2)
train_size = len(dataset) - val_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"学習: {train_size} サンプル / 検証: {val_size} サンプル")

# ----------------------------------------------------
# モデル・学習設定
# ----------------------------------------------------
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用デバイス: {device}")

model     = PTCGBaselineNet(input_dim=INPUT_DIM, max_actions=MAX_ACTIONS).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

# ----------------------------------------------------
# 学習ループ
# ----------------------------------------------------
print("\n== 学習開始 ==")
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    # --- 学習フェーズ ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for states, masks, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]", leave=False):
        states, masks, targets = states.to(device), masks.to(device), targets.to(device)

        outputs        = model(states)
        masked_outputs = outputs.masked_fill(masks, -1e9)
        loss           = criterion(masked_outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss    += loss.item() * states.size(0)
        preds          = masked_outputs.argmax(dim=1)
        train_correct += (preds == targets).sum().item()
        train_total   += states.size(0)

    # --- 検証フェーズ ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for states, masks, targets in val_loader:
            states, masks, targets = states.to(device), masks.to(device), targets.to(device)
            outputs        = model(states)
            masked_outputs = outputs.masked_fill(masks, -1e9)
            loss           = criterion(masked_outputs, targets)

            val_loss    += loss.item() * states.size(0)
            preds        = masked_outputs.argmax(dim=1)
            val_correct += (preds == targets).sum().item()
            val_total   += states.size(0)

    t_loss = train_loss / train_total
    v_loss = val_loss   / val_total
    t_acc  = train_correct / train_total * 100
    v_acc  = val_correct   / val_total   * 100

    print(f"Epoch {epoch+1:2d}/{EPOCHS}  "
          f"train loss={t_loss:.4f} acc={t_acc:.1f}%  |  "
          f"val loss={v_loss:.4f} acc={v_acc:.1f}%")

    scheduler.step(v_loss)

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"  → ベストモデル保存 (val_loss={v_loss:.4f})")

print(f"\n== 学習完了。モデル: {MODEL_PATH}  正規化パラメータ: {NORM_PATH} ==")


In [ ]:
import torch
import numpy as np

# ----------------------------------------------------
# 保存済みモデルの読み込みと推論テスト
# submission/main.py に組み込む際の参考コード
# ----------------------------------------------------

norm_data = np.load(NORM_PATH)
norm_mean = norm_data['mean']
norm_std  = norm_data['std']

inference_model = PTCGBaselineNet(input_dim=INPUT_DIM, max_actions=MAX_ACTIONS)
inference_model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
inference_model.eval()

print(f"モデル読み込み完了: {MODEL_PATH}")
print(f"正規化パラメータ読み込み完了: {NORM_PATH}")

def predict_action(state_vec: np.ndarray, valid_count: int) -> int:
    """
    state_vec   : extract_detailed_state_vector() の出力 (np.ndarray, shape=(INPUT_DIM,))
    valid_count : obs.select.option の長さ
    returns     : 選択するインデックス (0-indexed)
    """
    norm_state = (state_vec - norm_mean) / norm_std
    x = torch.tensor(norm_state, dtype=torch.float32).unsqueeze(0)

    mask = torch.arange(MAX_ACTIONS) >= valid_count
    mask = mask.unsqueeze(0)

    with torch.no_grad():
        logits = inference_model(x)
        masked = logits.masked_fill(mask, -1e9)
        action = masked.argmax(dim=1).item()

    return action


# --- 動作確認: アーカイブの1サンプルで推論 ---
import json, os
archive_dir = JSON_DIRECTORY
sample_files = sorted(os.listdir(archive_dir))[:1]
sample_path = os.path.join(archive_dir, sample_files[0])

with open(sample_path, 'r') as f:
    log = json.load(f)

print(f"\n推論テスト ({os.path.basename(sample_path)})")
for step in log['steps']:
    for ps in step:
        sv = extract_detailed_state_vector(ps, card_dict, MAX_CARD_ID)
        if sv is None:
            continue
        sel = ps.get('observation', {}).get('select')
        if not sel or 'option' not in sel:
            continue
        valid_count = len(sel['option'])
        if valid_count == 0:
            continue

        pred  = predict_action(sv, valid_count)
        truth = (ps.get('action') or [None])[0]
        print(f"  選択肢数={valid_count:3d}  予測={pred}  正解={truth}  {'✓' if pred == truth else '✗'}")
        break  # 1ステップだけ確認
